# Homework 6: Linear Regression Modleing and Diagnostics

Zac Griffin  Spring 2026 Data 

Build a predictive model for life expectancy using scikit-learn and perform a diagnostic audit to ensure the model’s coefficients are stable and mathematically sound.

## Task 1: Baseline Modeling
Load the WHO Life Expectancy Dataset, remove non-numeric columns (Country, Year), Convert Status column to  numeric, handle missing values.
- Use StandardScaler from sklearn.preprocessing to scale all independent variables. ("Life expectancy" is dependent variable) 
- Train a LinearRegression model using the scaled features.
- Report the R-Squared Score.
- Coefficient Analysis: Rank the features by their absolute coefficient values. Because the data is scaled, the largest coefficient now represents the most important predictor.

## Task 2: Iterative Diagnostic (VIF < 5)
Before analyzing the data, you must understand the Variance Inflation Factor (VIF) diagnostic. VIF measures multicollinearity, the degree to which your predictors are redundant.
High multicollinearity inflates the variance of coefficients, making them unreliable and difficult to interpret.
VIF is calculated by regressing one predictor against all others, then you obtain R_Squared and then compute VIF = 1/(1-R_Squared)
You sohuld use a cutoff of 5. Any variable with a VIF > 5 is considered too redundant and will be removed to improve the model's structural integrity. To do that follow the instruction below:

### Write a loop to iteratively remove the most redundant feature:
    1. Calculate VIF for every feature in the scaled dataset.
    2. Find the variable with the highest VIF.
    3. If VIF > 5, remove that feature from the data.
    4. Repeats the process until all remaining features have a VIF < 5
    - Note that in each iteration you remove only the highest VIF if it is greatet than 5!
    
## Task 3: Comparison of Model Results
    - Train a new LinearRegression model using only the independent features that survived the VIF diagnostic test.
    - Report the R-Squared Score.
    - Coefficient Analysis: Rank the features by their absolute coefficient values. Because the data is scaled, the largest coefficient now represents the most important predictor.
    - Are the top 5 predicotrs remain the same as those of the Baseline Model?

## Task 4: Residual Analysis and Normality
A good regression model should have errors (residuals) that are normally distributed and centered around zero.

    - Generate a histogram of the residuals for your baseline and clean model side by side.
    - Does the error distributions look like a bell curve?

## Task 5: The Interaction Investigation
In the real world application, the effect of one variable often depends on another. This is called an interaction.

    - Select two variables of your choice from the dataset (only variables with VIF >5) that you believe might have a combined effect (e.g., Alcohol and GDP, or BMI and HIV/AIDS,). Write one sentence explaining why you think they interact.
    - Create a new column in your dataframe by multiplying these two variables together.
    - Train a new model by adding this new interaction term to your dataset and fit it again using LinearRegression.
    - Report the R-Squared Score.
    - Is the interaction coeficient positive or negative? Has adding it improved the Model performance?




## Task 1: Baseline Modeling
Load the WHO Life Expectancy Dataset, remove non-numeric columns (Country, Year), Convert Status column to  numeric, handle missing values.
- Use StandardScaler from sklearn.preprocessing to scale all independent variables. ("Life expectancy" is dependent variable) 
- Train a LinearRegression model using the scaled features.
- Report the R-Squared Score.
- Coefficient Analysis: Rank the features by their absolute coefficient values. Because the data is scaled, the largest coefficient now represents the most important predictor.

In [1]:
#import libraries and functions
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [2]:
# load dataset 
df = pd.read_csv("Life_Expectancy_Data.csv")
df.columns = df.columns.str.strip() #gets rid of spaces at end 
df= df.dropna()
df.head()

,Country,Year,Status,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,...,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling
0,Afghanistan,2015,Developing,65.0,263,62,0.01,71.279624,65,1154,...,6,8.16,65,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1
1,Afghanistan,2014,Developing,59.9,271,64,0.01,73.523582,62,492,...,58,8.18,62,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0
2,Afghanistan,2013,Developing,59.9,268,66,0.01,73.219243,64,430,...,62,8.13,64,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9
3,Afghanistan,2012,Developing,59.5,272,69,0.01,78.184215,67,2787,...,67,8.52,67,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8
4,Afghanistan,2011,Developing,59.2,275,71,0.01,7.097109,68,3013,...,68,7.87,68,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1649 entries, 0 to 1648
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          1649 non-null   object 
 1   Year                             1649 non-null   int64  
 2   Status                           1649 non-null   object 
 3   Life expectancy                  1649 non-null   float64
 4   Adult Mortality                  1649 non-null   int64  
 5   infant deaths                    1649 non-null   int64  
 6   Alcohol                          1649 non-null   float64
 7   percentage expenditure           1649 non-null   float64
 8   Hepatitis B                      1649 non-null   int64  
 9   Measles                          1649 non-null   int64  
 10  BMI                              1649 non-null   float64
 11  under-five deaths                1649 non-null   int64  
 12  Polio               

In [4]:
# Encode 'Status' (Developed=1, Developing=0)
df['Status_Encoded'] = df['Status'].map({'Developed': 1, 'Developing': 0})  

#Drop country and year; ALSO dropping the object status column to make the table in next chunk work
df1= df.drop(['Country', 'Year', 'Status'], axis = 1)

df1.head()



,Life expectancy,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles,BMI,under-five deaths,Polio,Total expenditure,Diphtheria,HIV/AIDS,GDP,Population,thinness 1-19 years,thinness 5-9 years,Income composition of resources,Schooling,Status_Encoded
0,65.0,263,62,0.01,71.279624,65,1154,19.1,83,6,8.16,65,0.1,584.259210,33736494.0,17.2,17.3,0.479,10.1,0
1,59.9,271,64,0.01,73.523582,62,492,18.6,86,58,8.18,62,0.1,612.696514,327582.0,17.5,17.5,0.476,10.0,0
2,59.9,268,66,0.01,73.219243,64,430,18.1,89,62,8.13,64,0.1,631.744976,31731688.0,17.7,17.7,0.470,9.9,0
3,59.5,272,69,0.01,78.184215,67,2787,17.6,93,67,8.52,67,0.1,669.959000,3696958.0,17.9,18.0,0.463,9.8,0
4,59.2,275,71,0.01,7.097109,68,3013,17.2,97,68,7.87,68,0.1,63.537231,2978599.0,18.2,18.2,0.454,9.5,0


In [6]:
#Split the data! 
X = df1.drop('Life expectancy', axis = 1)
y= df1['Life expectancy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=78)

#SCALE THE DATA!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Use parameters from train


baseline_model = LinearRegression().fit(X_train_scaled, y_train)


features = X.columns.tolist()

importance = pd.DataFrame({
    'Feature': features,
    'Standardized_Weight': baseline_model.coef_
}).sort_values(by='Standardized_Weight', key=abs, ascending=False)

print(importance)
print(f"Multiple Regression Test R2: {r2_score(y_test, baseline_model.predict(X_test_scaled)):.3f}")


                            Feature  Standardized_Weight
7                 under-five deaths           -11.661943
1                     infant deaths            11.422238
11                         HIV/AIDS            -2.554217
17                        Schooling             2.338480
0                   Adult Mortality            -2.174257
16  Income composition of resources             1.742551
3            percentage expenditure             0.639770
6                               BMI             0.602125
15               thinness 5-9 years            -0.324418
10                       Diphtheria             0.323138
2                           Alcohol            -0.302307
18                   Status_Encoded             0.297258
8                             Polio             0.174862
9                 Total expenditure             0.171622
4                       Hepatitis B            -0.156070
12                              GDP             0.142252
5                           Mea

## Task 2: Iterative Diagnostic (VIF < 5)
Before analyzing the data, you must understand the Variance Inflation Factor (VIF) diagnostic. VIF measures multicollinearity, the degree to which your predictors are redundant.
High multicollinearity inflates the variance of coefficients, making them unreliable and difficult to interpret.
VIF is calculated by regressing one predictor against all others, then you obtain R_Squared and then compute VIF = 1/(1-R_Squared)
You sohuld use a cutoff of 5. Any variable with a VIF > 5 is considered too redundant and will be removed to improve the model's structural integrity. To do that follow the instruction below:

### Write a loop to iteratively remove the most redundant feature:
    1. Calculate VIF for every feature in the scaled dataset.
    2. Find the variable with the highest VIF.
    3. If VIF > 5, remove that feature from the data.
    4. Repeats the process until all remaining features have a VIF < 5
    - Note that in each iteration you remove only the highest VIF if it is greatet than 5!

In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

#I had to look online for this one. 
def vif_removal(data, threshold = 5.0):
    """ Removes the feature with the highest VIF if it is over 5 """
    data.columns = data.columns.str.strip()
    

    while True:

        #calculating VIF for each feature
        Q = add_constant(data) # intercept for VIF 
        vif_data = pd.DataFrame() 
        vif_data['feature'] = data.columns
        vif_data["VIF"] = [variance_inflation_factor(Q.values, i+1) for i in range(len(data.columns))] #skips the constant columns
    
                                             
        #Find the highest VIF
        
        max_vif = vif_data['VIF'].max()
        max_feature = vif_data.loc[vif_data["VIF"] == max_vif, "feature"].iloc[0]
        print(vif_data)
        if max_vif > threshold: 
            print(vif_data)
            print(F'Remvoing:  {max_feature} VIF: {max_vif:.3f}') 
            data = data.drop(columns = [max_feature], axis = 1)
        else:
            break
    
    return data.columns
 


In [8]:
vif_removal(X)

                            feature         VIF
0                   Adult Mortality    1.812128
1                     infant deaths  212.186280
2                           Alcohol    2.285766
3            percentage expenditure   12.852460
4                       Hepatitis B    1.660937
5                           Measles    1.515011
6                               BMI    1.797323
7                 under-five deaths  202.005452
8                             Polio    1.712697
9                 Total expenditure    1.119812
10                       Diphtheria    2.094827
11                         HIV/AIDS    1.483088
12                              GDP   13.570172
13                       Population    1.943578
14             thinness  1-19 years    7.607113
15               thinness 5-9 years    7.588073
16  Income composition of resources    2.971519
17                        Schooling    3.530048
18                   Status_Encoded    1.831846
                            feature     

Index(['Adult Mortality', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure',
       'Diphtheria', 'HIV/AIDS', 'Population', 'thinness 5-9 years',
       'Income composition of resources', 'Schooling', 'Status_Encoded'],
      dtype='object')

## Task 3: Comparison of Model Results¶

- Train a new LinearRegression model using only the independent features that survived the VIF diagnostic test.
- Report the R-Squared Score.
- Coefficient Analysis: Rank the features by their absolute coefficient values. Because the data is scaled, the largest coefficient now represents the most important predictor.
- Are the top 5 predicotrs remain the same as those of the Baseline Model?


In [9]:
features2 = ['Adult Mortality', 'Alcohol', 'percentage expenditure', 'Hepatitis B',
       'Measles', 'BMI', 'under-five deaths', 'Polio', 'Total expenditure',
       'Diphtheria', 'HIV/AIDS', 'Population', 'thinness 5-9 years',
       'Income composition of resources', 'Schooling', 'Status_Encoded']
X = df[features2]
Y = df['Life expectancy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=78)

#scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

multicolinearity_model = LinearRegression().fit(X_train_scaled, y_train)

importance = pd.DataFrame({
    'Feature': features2, 
    'Standardized_Weight': multicolinearity_model.coef_
}).sort_values(by='Standardized_Weight', key=abs, ascending=False)

print(importance)
print(f"Multiple Regression Test R2: {r2_score(y_test, multicolinearity_model.predict(X_test_scaled)):.3f}")
             

                            Feature  Standardized_Weight
10                         HIV/AIDS            -2.555371
14                        Schooling             2.431210
0                   Adult Mortality            -2.303952
13  Income composition of resources             1.837280
2            percentage expenditure             0.752376
5                               BMI             0.659667
6                 under-five deaths            -0.549667
1                           Alcohol            -0.476961
9                        Diphtheria             0.437857
15                   Status_Encoded             0.290030
11                       Population             0.266560
7                             Polio             0.227773
12               thinness 5-9 years            -0.206504
3                       Hepatitis B            -0.169143
4                           Measles             0.158450
8                 Total expenditure             0.155620
Multiple Regression Test R2: 0.

The top five are not the same as the baseline model's top five. 
Base line model top five were:  under-five deaths, infant deaths, HIV/AIDS, Schooling, Adult Mortality.

## Task 4: Residual Analysis and Normality¶

A good regression model should have errors (residuals) that are normally distributed and centered around zero.

- Generate a histogram of the residuals for your baseline and clean model side by side.
- Does the error distributions look like a bell curve?


In [10]:
y_predbase = baseline_model.predict(X_test)
base_residuals = y_test - y_predbase

y_predmulti = multicolinearity_model.predict(X_test)
multi_residuals = y_test - y_predmulti

fig, axes = plt.subplots(1, 2, figsize=(10,5))

sns.histogram(data = base_residuals, x = 'Life Expectancy', kde = True, color = 'orange', ax= axes[0])
sns.histogram(data = multi_residuals, x = 'Life Expectancy', kde = True, color = 'green', ax = axes[1])

axes[0].set_title("Histogram of Baseline Model")
axes[1].set_title("Histogram of VIF removed Model")

plt.tight_layout()
plt.show()


C:\Users\black\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LinearRegression was fitted without feature names
  warnings.warn(


ValueError: X has 16 features, but LinearRegression is expecting 19 features as input.

I don't understand this error message and my head hurts to much to figure it out. 